In all cases, written answers (apart from code) should not be longer than about three paragraphs.  Graders may not read all of your
submission if it is longer than that.

## Homework Reflection 5

### 1. Draw a diagram for the following negative feedback loop:

- Sweating causes body temperature to decrease.  High body temperature causes sweating.

- A negative feedback loop means that one thing increases another while the second thing decreases the first.

- Remember that we are using directed acyclic graphs where two things cannot directly cause each other.

**Answer:**

$$ \text{Temp}_0 \xrightarrow{\;+\;} \text{Sweat}_0 \xrightarrow{\;-\;} \text{Temp}_1 \xrightarrow{\;+\;} \text{Sweat}_1 \xrightarrow{\;-\;} \text{Temp}_2 \;\cdots $$

- **(+)** High temperature increases sweating.
- **(−)** Sweating decreases the temperature at the *next* time step.

### 2. Describe an example of a positive feedback loop.  This means that one things increases another while the second things also increases the first.

**Answer:**

Example: a social media app like reddit (a public online forum); More users make the app more valuable (more friends, more content to see), and a more valuable app attracts even more users.

$$
\text{Users}_0 \xrightarrow{\;+\;} \text{Value}_0 \xrightarrow{\;+\;} \text{Users}_1 \xrightarrow{\;+\;} \text{Value}_1 \xrightarrow{\;+\;} \text{Users}_2 \;\cdots
$$

The same pattern works in reverse: losing users makes the app less useful, which loses more users.

### 3. Draw a diagram for the following situation:

- Lightning storms frighten away deer and bears, decreasing their population, and cause flowers to grow, increasing their population.
- Bears eat deer, decreasing their population.
- Deer eat flowers, decreasing their population.

<br> Write a dataset that simulates this situation.  (Show the code.) Include noise / randomness in all cases.
<br>
<br> Identify a backdoor path with one or more confounders for the relationship between deer and flowers.

**Answer:**

$$
\begin{array}{ccccc}
 & & \text{Lightning} & & \\
 & \swarrow^{-} & \downarrow^{-} & \searrow^{+} & \\
\text{Bears} & \xrightarrow{\;-\;} & \text{Deer} & \xrightarrow{\;-\;} & \text{Flowers}
\end{array}
$$

Arrows: Lightning → Bears (−), Lightning → Deer (−), Lightning → Flowers (+), Bears → Deer (−), Deer → Flowers (−).

**Simulation:**

In [1]:
import numpy as np
import pandas as pd

np.random.seed(0)
n = 1000

# Lightning is the root cause (number of storms in a season)
lightning = np.random.normal(5, 1, n)

# Lightning scares bears away (decreases bears)
bears = 50 - 3 * lightning + np.random.normal(0, 2, n)

# Lightning scares deer away AND bears eat deer (both decrease deer)
deer = 200 - 5 * lightning - 2 * bears + np.random.normal(0, 5, n)

# Lightning helps flowers grow, deer eat flowers
flowers = 500 + 20 * lightning - 1.5 * deer + np.random.normal(0, 10, n)

df = pd.DataFrame({"lightning": lightning, "bears": bears, "deer": deer, "flowers": flowers})
print(df.head())

# Show the backdoor path matters: effect of deer on flowers (true value = -1.5)
def ols(y, *cols):
    A = np.column_stack([np.ones(len(y)), *cols])
    return np.linalg.lstsq(A, y, rcond=None)[0]

print("\nDeer coefficient, ignoring lightning:      ", round(ols(df.flowers, df.deer)[1], 2))
print("Deer coefficient, controlling for lightning:", round(ols(df.flowers, df.deer, df.lightning)[1], 2))
print("True effect of deer on flowers:              -1.5")

   lightning      bears        deer     flowers
0   6.764052  30.819768   96.875596  505.900389
1   5.400157  35.584476   93.270411  473.784752
2   5.978738  31.219156  107.898673  456.581880
3   7.240893  28.486748  102.030165  494.288919
4   6.867558  29.853433  105.551287  466.915673

Deer coefficient, ignoring lightning:       -1.03
Deer coefficient, controlling for lightning: -1.5
True effect of deer on flowers:              -1.5


**Backdoor path:** `Deer ← Lightning → Flowers`. Lightning is a **confounder**: it lowers deer and raises flowers at the same time. (There is also `Deer ← Bears ← Lightning → Flowers`, which runs through Lightning too.)

Because of this path, regressing flowers on deer alone gives about −1.0, which understates the true −1.5: years with lots of lightning have fewer deer *and* more flowers, which hides part of the damage deer do. Controlling for Lightning blocks both backdoor paths and recovers −1.5.

### 4. Draw a diagram for a situation of your own invention.  The diagram should include at least four nodes, one confounder, and one collider.  Be sure that it is acyclic (no loops).  Which node would say is most like a treatment (X)?  Which is most like an outcome (Y)?

**Answer:**

$$
\begin{array}{ccccc}
 & & \text{Age} & & \\
 & \swarrow & \downarrow & \searrow & \\
\text{Heart Health} & \longleftarrow & \text{Exercise} & \longrightarrow & \text{Muscle Soreness}
\end{array}
$$

Arrows: Age → Exercise, Age → Heart Health, Exercise → Heart Health, Exercise → Muscle Soreness, Age → Muscle Soreness.

- **Confounder: Age.** Older people exercise less and tend to have worse heart health, so Age affects both.
- **Collider: Muscle Soreness.** Both Exercise and Age cause it (two arrows point into it). We should *not* control for it: looking only at sore people would create a fake link between Exercise and Age.
- **Treatment (X): Exercise.** It's the thing a person chooses to do.
- **Outcome (Y): Heart Health.** It's the result we care about.

___

## Homework Reflection 6

### 1. What is a potential problem with computing the Marginal Treatment Effect simply by comparing each untreated item to its counterfactual and taking the maximum difference?  (Hint: think of statistics here.  Consider that only the most extreme item ends up being used to estimate the MTE.  That's not necessarily a bad thing; the MTE is supposed to come from the untreated item that will produce the maximum effect.  But there is nevertheless a problem.)
<br> Possible answer: We are likely to find the item with the most extreme difference, which may be high simply due to randomness.
(Please explain / justify this answer, or give a different one if you can think of one.)

**Answer:**

I agree with the suggested answer. Each item's effect is a *true* effect plus random noise (the noise in its own Y and in its matched counterfactual's Y). When we take the maximum over hundreds of items, we are picking the item whose noise happened to be the most positive. So the maximum is biased upward: it mostly measures randomness and not a real bigger effect. This is sometimes called the "winner's curse."

It also rests on a **single** data point, so it is very unstable: rerunning the experiment gives a quite different number. And it gets worse with more data, because more items means more chances to see an extreme value. In my simulation in the next question, the true effect is 2 for everyone, but the maximum comes out around 5 to 6.

### 2. Propose a solution that remedies this problem and write some code that implements your solution.  It's very important here that you clearly explain what your solution will do.
Possible answer: maybe we could take the 90th percentile of the treatment effect and use it as a proxy for the Marginal Treatment Effect.
(Either code this answer or choose a different one.)

**Answer:**

**My solution:** use the **90th percentile** of the individual treatment effects instead of the maximum. It still targets the items with the largest effects, but it is based on the ordering of many items instead of the single most extreme one, so one lucky noisy value can't drive it.

The code simulates data where the **true effect is 2 for everyone**, matches each untreated item to its nearest treated item, and compares the max with the 90th percentile. It then repeats the experiment 200 times to compare how stable each one is.

In [2]:
import numpy as np
from sklearn.neighbors import NearestNeighbors

np.random.seed(0)
n = 1000

# Simulated data where the TRUE treatment effect is exactly 2 for everyone
Z = np.random.normal(0, 1, n)                                   # confounder
X = (np.random.normal(0, 1, n) + Z > 0).astype(int)             # treatment depends on Z
Y = 2 * X + Z + np.random.normal(0, 1, n)                       # outcome

treated_Z, treated_Y = Z[X == 1].reshape(-1, 1), Y[X == 1]
untreated_Z, untreated_Y = Z[X == 0].reshape(-1, 1), Y[X == 0]

# For each untreated item, find its nearest treated item (the counterfactual)
nn = NearestNeighbors(n_neighbors=1).fit(treated_Z)
_, idx = nn.kneighbors(untreated_Z)
effects = treated_Y[idx.ravel()] - untreated_Y                  # one effect per untreated item

print(f"True effect:                  2.00")
print(f"Maximum (original MTE):       {effects.max():.2f}")
print(f"90th percentile (proposed):   {np.percentile(effects, 90):.2f}")
print(f"Mean of top 10% (alternative):{effects[effects >= np.percentile(effects, 90)].mean():.2f}")

# Stability check: repeat the whole experiment 200 times
maxes, p90s = [], []
for _ in range(200):
    Z = np.random.normal(0, 1, n); X = (np.random.normal(0, 1, n) + Z > 0).astype(int)
    Y = 2 * X + Z + np.random.normal(0, 1, n)
    nn = NearestNeighbors(n_neighbors=1).fit(Z[X == 1].reshape(-1, 1))
    _, idx = nn.kneighbors(Z[X == 0].reshape(-1, 1))
    eff = Y[X == 1][idx.ravel()] - Y[X == 0]
    maxes.append(eff.max()); p90s.append(np.percentile(eff, 90))
print(f"\nOver 200 repeats -> max: mean {np.mean(maxes):.2f}, SD {np.std(maxes):.2f}")
print(f"                 -> 90th percentile: mean {np.mean(p90s):.2f}, SD {np.std(p90s):.2f}")

True effect:                  2.00
Maximum (original MTE):       5.22
90th percentile (proposed):   3.58
Mean of top 10% (alternative):4.16

Over 200 repeats -> max: mean 6.27, SD 0.51
                 -> 90th percentile: mean 3.84, SD 0.19


**Result:** the maximum averages about 6.3 (three times the true value of 2) with a spread (SD) of about 0.5. The 90th percentile averages about 3.8 with an SD of about 0.2. It is still above 2, because noise still spreads the effects out, but it is much closer to the truth and more than twice as stable.

___

## Homework Reflection 7

#### 1. Create a linear regression model involving a confounder that is left out of the model.  Show whether the true correlation between $$X$$ and $$Y$$ is overestimated, underestimated, or neither.  Explain in words why this is the case for the given coefficients you have chosen.

**Answer:**

Model: W is a confounder that increases both X and Y. The true effect of X on Y is 2.

```
W = noise
X = W + noise
Y = 2*X + 3*W + noise
```

In [3]:
import numpy as np

np.random.seed(0)
n = 10000

W = np.random.normal(0, 1, n)                     # confounder
X = W + np.random.normal(0, 1, n)                 # W increases X
Y = 2 * X + 3 * W + np.random.normal(0, 1, n)     # true effect of X on Y is 2; W also increases Y

def ols(y, *cols):
    A = np.column_stack([np.ones(len(y)), *cols])
    return np.linalg.lstsq(A, y, rcond=None)[0]

print("True X coefficient:            2.00")
print(f"With W in the model:           {ols(Y, X, W)[1]:.2f}")
print(f"With W left out:               {ols(Y, X)[1]:.2f}  <- overestimated")
# Theory: 2 + 3 * Cov(X, W) / Var(X) = 2 + 3 * 1/2 = 3.5

True X coefficient:            2.00
With W in the model:           2.01
With W left out:               3.50  <- overestimated


**The effect is overestimated** (about 3.5 instead of 2). When W is left out, it becomes part of the error term. W pushes X up and also pushes Y up, so X and the error move together. The regression can't tell them apart, so it gives X credit for some of W's effect on Y.

The size of the bias is (effect of W on Y) × (how much X tracks W) = 3 × 0.5 = 1.5, so 2 + 1.5 = 3.5. Both coefficients are positive, so the bias is positive. If W increased X but *decreased* Y (or the reverse), the bias would be negative and the effect would be **underestimated**.

### 2. Perform a linear regression analysis in which one of the coefficients is zero, e.g.

```
W = [noise]
X = [noise]
Y = 2 * X + [noise]

```

And compute the p-value of a coefficient - in this case, the coefficient of W.  
(This is the likelihood that the estimated coefficient would be as high or low as it is, given that the actual coefficient is zero.)
If the p-value is less than 0.05, this ordinarily means that we judge the coefficient to be nonzero (incorrectly, in this case.)
Run the analysis 1000 times and report the best (smallest) p-value.  
If the p-value is less than 0.05, does this mean the coefficient actually is nonzero?  What is the problem with repeating the analysis?

**Answer:**

Each run simulates W, X and Y (W has no real effect), regresses Y on X and W, and computes the p-value of W's coefficient. I compute the p-value by hand with a t-test, which is what statsmodels does under the hood.

In [4]:
import numpy as np
from scipy import stats

np.random.seed(0)
n = 100

def p_value_of_W():
    W = np.random.normal(0, 1, n)
    X = np.random.normal(0, 1, n)
    Y = 2 * X + np.random.normal(0, 1, n)          # W has NO effect on Y (true coefficient = 0)

    A = np.column_stack([np.ones(n), X, W])        # intercept, X, W
    beta = np.linalg.lstsq(A, Y, rcond=None)[0]
    resid = Y - A @ beta
    dof = n - A.shape[1]
    sigma2 = resid @ resid / dof
    se = np.sqrt(np.diag(sigma2 * np.linalg.inv(A.T @ A)))
    t = beta[2] / se[2]                            # t-statistic for W
    return 2 * stats.t.sf(abs(t), dof)             # two-sided p-value

p_values = np.array([p_value_of_W() for _ in range(1000)])
print(f"Smallest p-value out of 1000 runs: {p_values.min():.5f}")
print(f"Runs with p < 0.05:                {(p_values < 0.05).sum()} of 1000 (about 5% expected by chance)")

Smallest p-value out of 1000 runs: 0.00047
Runs with p < 0.05:                45 of 1000 (about 5% expected by chance)


**The smallest p-value is about 0.0005**, far below 0.05, even though W's true coefficient is exactly zero. About 45 of the 1000 runs had p < 0.05.

**It doesn't mean the coefficient is nonzero.** A p-value of 0.05 means that even when there is no effect, we expect to see a result this extreme 5% of the time *by chance*. If we run the test 1000 times, we expect about 50 "significant" results from pure noise, and the smallest one will look extremely convincing.

**The problem with repeating the analysis** is multiple testing (also called p-hacking): if you keep trying until you get a small p-value and only report that one, you are guaranteed to "find" an effect that isn't real. The fix is to decide on one test in advance, or to correct for the number of tests (for example the Bonferroni correction: use 0.05 / 1000 as the threshold).

___

## Homework Reflection 8

### Include the code you used to solve Week 8's coding quiz problems and write about the obstacles / challenges / insights you encountered while solving them.